# C-ODT: a first look

This notebook fits C-ODT droplet size distributions with `dsdfit` and compares the results with C-ODT's own supersaturation statistics.

For each run, it does two things:

1. **Fit.** Build a histogram from the hour-2 DSD output, pooling snapshots spaced a few minutes apart, and estimate $D_x$ and $\bar s$. This is done four ways:
   - with 20 snapshots, 3 minutes apart, or with 12 snapshots, 5 minutes apart;
   - with the 1 µm diameter cutoff, or with all bins.
2. **Predict.** Compute the supersaturation statistics:
   - $\sigma_s$ and $\tau$, giving the prediction $D_x \approx a^2\sigma_s^2\tau$;
   - the mean supersaturation the droplets experience, to compare with the fitted $\bar s$.

   These come from the droplet trajectories (what the droplets experience, every 0.1 s). The same statistics are also computed from the supersaturation field at fixed heights, for comparison.

**Data.** The C-ODT output is not part of this repository. Set `DATA_DIR` below to the folder that holds the runs, one subfolder per run, each with `output/<run>.nc` and `output/<run>_particles.nc`. Reading the files needs the `netCDF4` package (`pip install netCDF4`).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import netCDF4
from dsdfit import Conditions, Histogram, fit, model_fractions, predicted_D_x
from dsdfit.plots import log_diameter_axis

DATA_DIR = Path("path/to/Sidewall_Data")          # folder containing the C-ODT runs
RUNS = ["SWS95_C", "SWS97_C", "SWS100_C"]  # sidewall relative humidity 95%, 97%, 100%
HOUR_2 = (3600.0, 7200.0)                  # steady period used for everything below [s]

## 1. Reading a run

The main output file has the DSD (counts of droplets per **radius** bin, at every output time), the temperature and the supersaturation (in %) as functions of height and time. The dry particle radius is read from the particles file.

In [ ]:
def load_run(name):
    """The pieces of a C-ODT run that the fit needs."""
    ds = netCDF4.Dataset(DATA_DIR / name / "output" / f"{name}.nc")
    t = ds["time"][:].astype(float)
    in_hour_2 = (t >= HOUR_2[0]) & (t <= HOUR_2[1])
    run = dict(
        name=name,
        time=t,
        DSD=ds["DSD"][:],                                        # (time, radius bin), counts
        radius_edges=ds["radius_edges"][:].astype(float),        # um
        T_mean=float(ds["T"][:][in_hour_2].mean()) + 273.15,     # K, domain average over hour 2
        p=float(ds.getncattr("PARAMETERS.pres")),                # Pa
        H=float(ds.getncattr("PARAMETERS.H")),                   # m
    )
    particles = netCDF4.Dataset(DATA_DIR / name / "output" / f"{name}_particles.nc")
    run["d_dry"] = 2e6 * float(particles["solute_radius"][:].mean())   # dry diameter, um
    return run


def conditions_for(run):
    return Conditions(T=run["T_mean"], p=run["p"], d_dry=run["d_dry"], H=run["H"])


def pooled_histogram(run, times, cutoff_diameter=None):
    """Add up the DSD at the output times nearest to `times`, as a dsdfit Histogram.

    Radius bins become diameter bins (x 2). With a cutoff, bins whose lower edge is
    below it are dropped. Empty bins beyond the largest droplets are dropped too
    (there are no droplets there, and it keeps the solver fast).
    """
    idx = [int(np.argmin(np.abs(run["time"] - t))) for t in times]
    counts = run["DSD"][idx].sum(axis=0)
    edges = 2.0 * run["radius_edges"]
    first = 0 if cutoff_diameter is None else int(np.nonzero(edges[:-1] >= cutoff_diameter)[0][0])
    last = min(int(np.nonzero(counts)[0][-1]) + 3, len(counts))
    return Histogram(edges[first:last + 1], counts[first:last])

## 2. Supersaturation statistics

With $s'$ the fluctuation about the mean, $\sigma_s^2$ is the average of $s'^2$, and the autocorrelation is

$$\rho(t) = \frac{\text{average of } s'(t_0)\,s'(t_0+t)}{\sigma_s^2},$$

averaged over all droplets (trajectories) or all heights (field). The time scale is $\tau = \int_0^\infty \rho(t)\,dt$, integrated with the trapezoid rule up to the first zero crossing of $\rho$.

- **Trajectories.** $s'$ is measured from the mean over all droplets and times, because a droplet moving between moister and drier regions experiences that as a fluctuation.
- **Field.** $s'$ is measured from the time-mean at each height. Otherwise a fixed point's offset from the overall mean would never decorrelate.

In [ ]:
def autocorrelation(X, present, max_lag):
    """Average of X[t] * X[t + lag] over all columns and times where both are present (via FFT)."""
    n_t = X.shape[0]
    nfft = 1 << int(np.ceil(np.log2(2 * n_t)))
    num, pairs = np.zeros(max_lag + 1), np.zeros(max_lag + 1)
    for c in range(0, X.shape[1], 500):
        x = np.fft.rfft(X[:, c:c + 500].astype(float), nfft, axis=0)
        m = np.fft.rfft(present[:, c:c + 500].astype(float), nfft, axis=0)
        num += np.fft.irfft(x * np.conj(x), nfft, axis=0)[:max_lag + 1].sum(axis=1)
        pairs += np.fft.irfft(m * np.conj(m), nfft, axis=0)[:max_lag + 1].sum(axis=1)
    return num / np.round(pairs)


def integral_time_scale(rho, dt):
    """Area under rho up to its first zero crossing (trapezoid rule)."""
    k = int(np.argmax(rho <= 0)) if np.any(rho <= 0) else len(rho) - 1
    r = rho[:k + 1]
    return dt * (0.5 * r[0] + r[1:-1].sum() + 0.5 * r[-1])


def stats_from_trajectories(name, max_lag_s=150.0):
    """Mean supersaturation, sigma_s and tau experienced by droplets (particles file)."""
    p = netCDF4.Dataset(DATA_DIR / name / "output" / f"{name}_particles.nc")
    t = p["time"][:].astype(float)
    dt = float(np.median(np.diff(t)))
    rows = p["row_sizes"][:].astype(int)
    s = p["supersaturation"][:].astype(float) / 100.0                # % -> fraction
    step = np.repeat(np.arange(len(rows)), rows)
    _, column = np.unique(p["particle_id"][:], return_inverse=True)  # one column per droplet
    s_mean, sigma2 = s.mean(), s.var()
    X = np.zeros((len(rows), column.max() + 1), np.float32)
    present = np.zeros_like(X)
    X[step, column] = s - s_mean
    present[step, column] = 1.0
    rho = autocorrelation(X, present, int(max_lag_s / dt)) / sigma2
    return dict(s_mean=s_mean, sigma_s=np.sqrt(sigma2), tau=integral_time_scale(rho, dt), rho=rho, dt=dt,
                window=(t[0], t[-1]))


def stats_from_field(run, name, max_lag_s=300.0):
    """Mean supersaturation, sigma_s and tau at fixed heights over hour 2 (main output file)."""
    ds = netCDF4.Dataset(DATA_DIR / name / "output" / f"{name}.nc")
    t = run["time"]
    keep = np.nonzero((t >= HOUR_2[0]) & (t <= HOUR_2[1]))[0]
    keep = keep[np.unique(np.round(t[keep]), return_index=True)[1]]  # drop repeated output times
    S = ds["S"][:][keep].astype(float) / 100.0                       # (time, height), fraction
    dt = float(np.median(np.diff(t[keep])))
    fluct = S - S.mean(axis=0)
    sigma2 = fluct.var()
    rho = autocorrelation(fluct, np.ones_like(fluct), int(max_lag_s / dt)) / sigma2
    return dict(s_mean=S.mean(), sigma_s=np.sqrt(sigma2), tau=integral_time_scale(rho, dt), rho=rho, dt=dt,
                profile_sd=S.mean(axis=0).std())

## 3. Run everything

Snapshot spacing: droplets live about 2–3 minutes on average in these runs (number in the domain divided by the injection rate), so snapshots 3 or 5 minutes apart contain mostly different droplets.

In [ ]:
SNAPSHOT_SETS = {
    "20 snapshots, 3 min apart": HOUR_2[0] + 180.0 * np.arange(20),
    "12 snapshots, 5 min apart": HOUR_2[0] + 300.0 * np.arange(12),
}
CUTOFFS = {"1 um cutoff": 1.0, "all bins": None}

runs, conds, traj, field, fits = {}, {}, {}, {}, {}
for name in RUNS:
    runs[name] = run = load_run(name)
    conds[name] = cond = conditions_for(run)
    traj[name] = stats_from_trajectories(name)
    field[name] = stats_from_field(run, name)
    for set_name, times in SNAPSHOT_SETS.items():
        for cut_name, cut in CUTOFFS.items():
            hist = pooled_histogram(run, times, cut)
            fits[name, set_name, cut_name] = (hist, fit(hist, cond))
    print(f"{name}: done")

### Supersaturation statistics and predicted $D_x$

In [ ]:
print(f"{'run':10s} {'source':13s} {'mean s':>9s} {'sigma_s':>9s} {'tau (s)':>8s} {'predicted D_x':>14s}")
for name in RUNS:
    for label, st in [("trajectories", traj[name]), ("field", field[name])]:
        D_pred = predicted_D_x(st["sigma_s"], st["tau"], conds[name])
        print(f"{name:10s} {label:13s} {st['s_mean']:9.5f} {st['sigma_s']:9.5f} {st['tau']:8.2f} {D_pred:14.3f}")
    print(f"{'':10s} {'':13s} (field: sd of the time-mean profile = {field[name]['profile_sd']:.5f})")

In [ ]:
fig, axes = plt.subplots(1, len(RUNS), figsize=(4 * len(RUNS), 3.2), sharey=True)
for ax, name in zip(np.atleast_1d(axes), RUNS):
    for label, st, style in [("trajectories (0.1 s)", traj[name], "-"), ("field (1 s)", field[name], "o-")]:
        lags = st["dt"] * np.arange(len(st["rho"]))
        keep = lags <= 30
        ax.plot(lags[keep], st["rho"][keep], style, ms=3, label=label)
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set_title(name); ax.set_xlabel("time lag (s)")
np.atleast_1d(axes)[0].set_ylabel("autocorrelation of s"); np.atleast_1d(axes)[0].legend(frameon=False)
plt.show()

### Fits

In [ ]:
print(f"{'run':10s} {'snapshots':27s} {'bins':12s} {'n':>6s} {'D_x':>16s} {'s_bar':>20s} {'corr':>6s} {'J_min':>7s} {'J_exp':>9s}")
for (name, set_name, cut_name), (hist, r) in fits.items():
    print(f"{name:10s} {set_name:27s} {cut_name:12s} {hist.n:6d} {r.D_x:7.3f} +/- {r.D_x_err:5.3f} "
          f"{r.s_bar:9.5f} +/- {r.s_bar_err:7.5f} {r.correlation:6.2f} {r.J_min:7.0f} {r.J_expected:5.0f}+/-{r.J_spread:.0f}"
          + ("" if r.fit_ok else "  <- fit check fails"))

### Fitted versus predicted

Fitted values with their error bars, next to the predictions from the trajectory statistics (filled) and the field statistics (open).

In [ ]:
cases = [(s, c) for s in SNAPSHOT_SETS for c in CUTOFFS]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for i, name in enumerate(RUNS):
    for j, (set_name, cut_name) in enumerate(cases):
        r = fits[name, set_name, cut_name][1]
        x = i + 0.15 * (j - 1.5)
        axes[0].errorbar(x, r.D_x, yerr=r.D_x_err, fmt="s", color=f"C{j}", label=f"{set_name}, {cut_name}" if i == 0 else None)
        axes[1].errorbar(x, 100 * r.s_bar, yerr=100 * r.s_bar_err, fmt="s", color=f"C{j}")
    axes[0].plot(i + 0.4, predicted_D_x(traj[name]["sigma_s"], traj[name]["tau"], conds[name]), "k*", ms=11,
                 label="predicted (trajectories)" if i == 0 else None)
    axes[0].plot(i + 0.4, predicted_D_x(field[name]["sigma_s"], field[name]["tau"], conds[name]), "k*", mfc="none", ms=11,
                 label="predicted (field)" if i == 0 else None)
    axes[1].plot(i + 0.4, 100 * traj[name]["s_mean"], "k*", ms=11)
for ax, ylabel in zip(axes, ["D_x (µm⁴/s)", "s_bar (%)"]):
    ax.set_xticks(range(len(RUNS))); ax.set_xticklabels(RUNS); ax.set_ylabel(ylabel)
axes[0].set_ylim(bottom=0); axes[0].legend(frameon=False, fontsize=7)
plt.show()

### What the shapes look like

For each run (20 snapshots, all bins): the C-ODT histogram, the best fit, and the model evaluated at C-ODT's own statistics (trajectory $\sigma_s$, $\tau$ and mean supersaturation). The log scale on the right shows the small droplets.

In [ ]:
for name in RUNS:
    hist, r = fits[name, "20 snapshots, 3 min apart", "all bins"]
    st = traj[name]
    p_stats = model_fractions(predicted_D_x(st["sigma_s"], st["tau"], conds[name]), st["s_mean"], conds[name], hist.edges)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    for ax in axes:
        ax.plot(hist.centers, hist.fractions, "o", ms=3, color="0.3", label="C-ODT")
        ax.plot(hist.centers, r.fractions, "--", color="tab:orange", label=f"best fit (D_x={r.D_x:.2f}, s_bar={r.s_bar:.4f})")
        ax.plot(hist.centers, p_stats, "-", color="tab:blue", label="model at C-ODT statistics")
        log_diameter_axis(ax, hist.edges[0], hist.edges[-1]); ax.set_xlabel("droplet diameter (µm)")
    axes[1].set_yscale("log"); axes[1].set_ylim(1e-6, 0.1)
    axes[0].set_ylabel("fraction of droplets in bin"); axes[0].set_title(name); axes[0].legend(frameon=False, fontsize=8)
    plt.show()

## Things to look at

- Does the fit check pass? If not, the estimates and their error bars should not be taken at face value.
- Do the fitted $D_x$ and $\bar s$ agree with the predictions, and do they change with the cutoff or the number of snapshots?
- Where in size do the C-ODT histogram and the model differ?